# 11.24 POMDPs

A POMDP replaces hidden state with a belief distribution you can update.

Partial observability means the agent tracks a belief over hidden states, then acts using that belief instead of pretending observations are states. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1119
random.seed(SEED)
np.random.seed(SEED)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * float(reward)
        power = power * gamma
    return total


lesson_rewards = [1, 0, 2]
lesson_gamma = 0.9
lesson_return = discounted_return(lesson_rewards, lesson_gamma)
assert abs(lesson_return - 2.62) < 1e-12


ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["up", "right", "down", "left"]
DELTAS = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}


def make_grid_env(name, height, width, start, goal, walls=None, slip=0.0, wind=None, sparse=True, horizon=35, obs_noise=0.0):
    walls = set(walls or [])
    wind = wind or {}
    states = []
    state_index = {}
    for row in range(height):
        for col in range(width):
            cell = (row, col)
            if cell not in walls:
                state_index[cell] = len(states)
                states.append(cell)
    env = {
        "name": name,
        "height": height,
        "width": width,
        "start": start,
        "goal": goal,
        "walls": walls,
        "slip": float(slip),
        "wind": wind,
        "sparse": bool(sparse),
        "horizon": int(horizon),
        "states": states,
        "state_index": state_index,
        "n_states": len(states),
        "n_actions": 4,
        "obs_noise": float(obs_noise),
    }
    return env


def step_cell(env, cell, action, rng):
    actual = int(action)
    if rng.random() < env["slip"]:
        actual = int(rng.integers(0, 4))
    row_delta, col_delta = DELTAS[actual]
    row = cell[0] + row_delta
    col = cell[1] + col_delta
    pushed = env["wind"].get((cell[0], cell[1]), (0, 0))
    row = row + pushed[0]
    col = col + pushed[1]
    candidate = (max(0, min(env["height"] - 1, row)), max(0, min(env["width"] - 1, col)))
    if candidate in env["walls"]:
        candidate = cell
    reward = -0.02
    done = candidate == env["goal"]
    if done:
        reward = 1.0
    if not env["sparse"]:
        distance = abs(candidate[0] - env["goal"][0]) + abs(candidate[1] - env["goal"][1])
        reward = reward - 0.01 * distance
    return candidate, reward, done


def transition_table(env):
    n_states = env["n_states"]
    n_actions = env["n_actions"]
    transitions = np.zeros((n_states, n_actions, n_states), dtype=float)
    rewards = np.zeros((n_states, n_actions, n_states), dtype=float)
    rng = np.random.default_rng(SEED + env["n_states"])
    samples = 80
    for state_id, cell in enumerate(env["states"]):
        for action in range(n_actions):
            for _ in range(samples):
                next_cell, reward, done = step_cell(env, cell, action, rng)
                next_id = env["state_index"][next_cell]
                transitions[state_id, action, next_id] = transitions[state_id, action, next_id] + 1.0
                rewards[state_id, action, next_id] = rewards[state_id, action, next_id] + reward
            total = transitions[state_id, action].sum()
            mask = transitions[state_id, action] > 0
            rewards[state_id, action, mask] = rewards[state_id, action, mask] / transitions[state_id, action, mask]
            transitions[state_id, action] = transitions[state_id, action] / total
    return transitions, rewards


def build_f12_env_ladder():
    ladder = [
        make_grid_env("D1 two-state chain", 1, 2, (0, 0), (0, 1), horizon=6, sparse=False),
        make_grid_env("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), slip=0.15, horizon=10, sparse=False),
        make_grid_env("D3 4x4 gridworld", 4, 4, (0, 0), (3, 3), walls={(1, 1), (2, 1)}, horizon=25, sparse=False),
        make_grid_env("D4 stochastic windy grid", 5, 5, (0, 0), (4, 4), walls={(1, 2), (2, 2)}, slip=0.2, wind={(3, 1): (-1, 0), (3, 2): (-1, 0)}, horizon=35, sparse=False),
        make_grid_env("D5 larger sparse grid", 7, 7, (0, 0), (6, 6), walls={(1, 1), (1, 2), (2, 4), (3, 1), (3, 2), (4, 4), (5, 2)}, slip=0.25, wind={(5, 3): (-1, 0)}, horizon=45, sparse=True),
    ]
    assert len(ladder) == 5
    assert [env["n_states"] for env in ladder] == [2, 3, 14, 23, 42]
    return ladder


def greedy_rollout(env, policy, episodes=20):
    rng = np.random.default_rng(SEED + env["height"] + env["width"])
    returns = []
    paths = []
    for episode in range(episodes):
        cell = env["start"]
        rewards = []
        path = [cell]
        for step in range(env["horizon"]):
            state_id = env["state_index"][cell]
            action = int(policy[state_id])
            cell, reward, done = step_cell(env, cell, action, rng)
            rewards.append(reward)
            path.append(cell)
            if done:
                break
        returns.append(discounted_return(rewards, lesson_gamma))
        paths.append(path)
    return float(np.mean(returns)), paths[-1]


def plot_grid_values(ax, env, values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    for cell, idx in env["state_index"].items():
        grid[cell] = values[idx]
    image = ax.imshow(grid, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return image


## The concept, built once: Bayes belief update\n\nThe POMDP filter is\n$$b'(s')=\eta\,O(o\mid s',a)\sum_sP(s'\mid s,a)b(s).$$\nFor a two-state hidden chain, prediction $[0.5,0.5]$ and observation likelihood $[0.2,0.9]$ normalize to $[0.181818,0.818182]$.

In [ ]:

def belief_update(belief, transition, observation_likelihood):
    predicted = np.asarray(belief) @ np.asarray(transition)
    unnormalized = predicted * np.asarray(observation_likelihood)
    normalizer = unnormalized.sum()
    updated = unnormalized / normalizer
    return updated, predicted, float(normalizer)


belief = np.array([0.6, 0.4])
transition = np.array([[0.7, 0.3], [0.2, 0.8]])
obs_like = np.array([0.2, 0.9])
updated, predicted, normalizer = belief_update(belief, transition, obs_like)
print("predicted", predicted, "updated", np.round(updated, 6), "eta", round(1.0 / normalizer, 6))
assert np.allclose(predicted, np.array([0.5, 0.5]))
assert np.allclose(np.round(updated, 6), np.array([0.181818, 0.818182]))
assert abs(discounted_return([1, 0, 2], 0.9) - 2.62) < 1e-12


The reusable POMDP controller updates a grid belief from noisy local observations and chooses the action with highest belief-weighted value.

In [ ]:

def observe_cell(env, true_cell, rng):
    if rng.random() < env["obs_noise"]:
        return env["states"][int(rng.integers(0, env["n_states"]))]
    return true_cell


def grid_belief_update(env, belief, action, observation):
    transitions, rewards = transition_table(env)
    predicted = belief @ transitions[:, action, :]
    likelihood = np.ones(env["n_states"]) * (env["obs_noise"] / max(env["n_states"], 1))
    obs_id = env["state_index"][observation]
    likelihood[obs_id] = likelihood[obs_id] + 1.0 - env["obs_noise"]
    posterior = predicted * likelihood
    posterior = posterior / posterior.sum()
    return posterior


def belief_policy(env):
    transitions, rewards = transition_table(env)
    values = np.zeros(env["n_states"])
    for _ in range(50):
        q_values = np.sum(transitions * (rewards + 0.9 * values[None, None, :]), axis=2)
        values = np.max(q_values, axis=1)
    return q_values, values


def pomdp_rollout(env, episodes=20):
    rng = np.random.default_rng(SEED + env["n_states"])
    q_values, values = belief_policy(env)
    returns = []
    final_beliefs = []
    for _ in range(episodes):
        true_cell = env["start"]
        belief = np.ones(env["n_states"]) / env["n_states"]
        rewards_seen = []
        for step in range(env["horizon"]):
            action_scores = belief @ q_values
            action = int(np.argmax(action_scores))
            true_cell, reward, done = step_cell(env, true_cell, action, rng)
            observation = observe_cell(env, true_cell, rng)
            belief = grid_belief_update(env, belief, action, observation)
            rewards_seen.append(reward)
            if done:
                break
        returns.append(discounted_return(rewards_seen, 0.9))
        final_beliefs.append(belief)
    return float(np.mean(returns)), np.mean(final_beliefs, axis=0), values


## The dataset ladder

In [ ]:

ladder = build_f12_env_ladder()
noise_levels = [0.05, 0.10, 0.15, 0.25, 0.35]
for env, noise in zip(ladder, noise_levels):
    env["obs_noise"] = noise
for env in ladder:
    sample_states = env["states"][:5]
    print(env["name"], "states", env["n_states"], "obs_noise", env["obs_noise"], "sample", sample_states)


In [ ]:

results = []
artifacts = []
for env in ladder:
    mean_return, belief, values = pomdp_rollout(env, episodes=20)
    results.append({"rung": env["name"], "return": mean_return})
    artifacts.append((env, belief, values))
print("rung | return")
for row in results:
    print(row["rung"], round(row["return"], 3))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, belief, values) in enumerate(artifacts):
    plot_grid_values(axes[0, col], env, belief, env["name"])
axes[1, 0].plot([idx + 1 for idx in range(len(results))], [row["return"] for row in results], marker="o")
axes[1, 0].set_title("return vs partial observability")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: letting notation hide shapes

Treating observations as if they were states collapses uncertainty to a scalar. Explicit belief vectors preserve the hidden-state shape.

In [ ]:

d5 = ladder[-1]
wrong_observation_state_count = 1
correct_belief_shape = (d5["n_states"],)
mean_return, belief, values = pomdp_rollout(d5, episodes=20)
print("wrong observation-as-state count", wrong_observation_state_count)
print("fixed belief vector shape", correct_belief_shape)
print("belief sum", round(float(belief.sum()), 6), "return", round(mean_return, 3))
assert belief.shape == correct_belief_shape


## Evaluate it + Practice

- Metric: compare D1-D5 return against a no-skill baseline.
- Sanity check: D1 should solve by hand and match the exact assertion in the build-once cell.
- Ablation: turn off the key idea and verify the metric worsens on D4 or D5.
- Failure signal: curves that look good only because support, entropy, shape, or rollout bias was hidden.

Practice 1: change one ladder parameter and predict the metric direction before running.


Practice 2: increase observation noise and compare final belief entropy on D5.